# Author diarization parameter tuning

Use this notebook to try different window settings, stride values, similarity thresholds, and text limits without editing the script each time.

Run the cells in order. Then adjust the parameters in the tuning cell and re-run it.

In [2]:
from __future__ import annotations

from statsmodels.graphics.tukeyplot import results

from author_prediction.deep_stylometry_encoder import DeepStylometryEncoder
from author_prediction.pipeline_implementation import run_pipeline
from author_prediction.profile_tracker import AuthorProfileTracker
from author_prediction.reporting import format_run_summary, summarize_run
from author_prediction.segmenter import split_into_sentences


/Users/joseph/Documents/author-prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_limited_text(path: str, max_chars: int | None = None, max_sentences: int | None = None) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    if max_chars is not None and max_chars > 0:
        text = text[:max_chars]

    if max_sentences is not None and max_sentences > 0:
        sentences = split_into_sentences(text)
        text = " ".join(sentences[:max_sentences])

    return text


def print_progress(progress: float, processed: int, total: int) -> None:
    pct = int(progress * 100)
    print(f"Processing text: {pct}% ({processed}/{total} sentences)", end="\r")

In [8]:
# Tune these values and re-run the experiment cell.
DATA_PATH = "README.md"
MAX_CHARS = 100000
MAX_SENTENCES = None

context_window_size = 20
stride = 5
sim_threshold = 0.99
ema_alpha = 0.22
min_tokens_for_update = 15
merge_threshold = 0.95

text = load_limited_text(DATA_PATH, max_chars=MAX_CHARS, max_sentences=MAX_SENTENCES)
sentences = split_into_sentences(text)
print(f"Loaded {len(sentences)} sentences from {DATA_PATH}")
print(f"Using {len(text)} characters of input text")

Loaded 1 sentences from README.md
Using 443 characters of input text


In [5]:
encoder = DeepStylometryEncoder()

In [ ]:
tracker = AuthorProfileTracker(
    sim_threshold=sim_threshold,
    ema_alpha=ema_alpha,
    min_tokens_for_update=min_tokens_for_update,
)

In [11]:
result = run_pipeline(
    text,
    encoder=encoder,
    tracker=tracker,
    context_window_size=context_window_size,
    stride=stride,
    merge_threshold=merge_threshold,
    progress_callback=print_progress,
)

print("\n")
summary = summarize_run(result)
print(format_run_summary(summary))

if result.get("merge_events"):
    print("\nAuthor merge events:")
    for event in result["merge_events"]:
        print(
            f"  - Merged {event['merged_from']} into {event['kept']} (similarity={event['similarity']:.4f})"
        )

print("\nPer-sentence detail:")
assignments = result["assignments"]
for i, r in enumerate(assignments):
    changed = i > 0 and assignments[i]["author_id"] != assignments[i - 1]["author_id"]
    sim_str = f"{r['similarity']:.4f}" if r["similarity"] is not None else "  n/a "
    marker = "  <- author change" if changed else ""
    print(
        f"  [{i:>4}] {r['author_id']:<10} sim={sim_str} new={str(r['is_new_author']):<5}{marker}"
    )

Processing text: 100% (1/1 sentences)

Processed 1 sentence(s)
Authors before merge: 3  |  after merge: 1
No merges were needed.
Change points: 0 at sentence indices []
Similarity to matched author -- mean: 1.0000, min: 1.0000, max: 1.0000

Per-author breakdown:
  Author_1: 0 sentence(s) (0.0% of doc), first seen at None, last seen at 0
  Author_4: 0 sentence(s) (0.0% of doc), first seen at None, last seen at 500
  Author_8: 1 sentence(s) (100.0% of doc), first seen at 0, last seen at 0

Per-sentence detail:
  [   0] Author_8   sim=1.0000 new=False


In [12]:
result

{'assignments': [{'author_id': 'Author_8',
   'similarity': 1.0000000000000002,
   'is_new_author': False,
   'profile_updated': True}],
 'id_remap': {'Author_1': 'Author_1',
  'Author_4': 'Author_4',
  'Author_8': 'Author_8'},
 'profiles': [{'author_id': 'Author_1', 'sample_count': 1, 'last_seen': 0},
  {'author_id': 'Author_4', 'sample_count': 100, 'last_seen': 500},
  {'author_id': 'Author_8', 'sample_count': 2, 'last_seen': 0}],
 'merge_events': []}

In [ ]:
# Optional: try a quick sweep over a few parameter combinations.
# This is useful when you want to compare settings side-by-side.
for stride_value in [1, 2, 5, 10]:
    for sim_value in [0.90, 0.94, 0.97]:
        text_sample = load_limited_text(DATA_PATH, max_chars=MAX_CHARS, max_sentences=MAX_SENTENCES)
        tracker = AuthorProfileTracker(
            sim_threshold=sim_value,
            ema_alpha=ema_alpha,
            min_tokens_for_update=min_tokens_for_update,
        )
        encoder = DeepStylometryEncoder()
        result = run_pipeline(
            text_sample,
            encoder=encoder,
            tracker=tracker,
            context_window_size=context_window_size,
            stride=stride_value,
            merge_threshold=merge_threshold,
        )
        summary = summarize_run(result)
        print(f"stride={stride_value}, sim_threshold={sim_value}: {summary['num_final_authors']} final authors, {summary['num_sentences']} windows processed")